# SYNTRA | 00 Starter: load FIRMS fire data

NASA Space Apps 2026, challenge: **Harmonization of MODIS and VIIRS Hot Spots**.

This notebook downloads the FIRMS hotspot files straight from NASA, puts MODIS and VIIRS into one table with the same columns, and draws the first monthly counts. Nothing needs to be uploaded.

**How to use:** Runtime > Run all. To change the country or the years, edit the settings cell and run all again.

Run it from the `notebooks` folder of the repo, or in Colab. Team workspace (plans, messages, decisions): see the SYNTRA Claude Doc. Data source: [NASA FIRMS](https://firms.modaps.eosdis.nasa.gov/).

## 1. Settings

In [ ]:
# Country name exactly as FIRMS writes it in its file names (spaces become underscores)
COUNTRY = "Saudi_Arabia"

# Years to load. MODIS starts in 2000, VIIRS S-NPP in 2012, VIIRS NOAA-20 in 2018.
# The yearly files currently end at 2025; 2026 needs the FIRMS API and a MAP_KEY.
YEARS = list(range(2000, 2026))

# FIRMS folder name -> label used in this notebook
INSTRUMENTS = {
    "modis": "MODIS",
    "viirs-snpp": "VIIRS_SNPP",
    "viirs-jpss1": "VIIRS_NOAA20",
}

# Keep only presumed vegetation fires (type 0)? Type 2 = static sources such as gas flares.
VEGETATION_ONLY = True

## 2. Download the yearly files from FIRMS

In [ ]:
import os, sys, pathlib, requests

# Files go to data/raw in the repo (or /content/firms_data in Colab).
# Set FIRMS_LOCAL_DIR to use files you already have without downloading.
IN_COLAB = "google.colab" in sys.modules
DEFAULT_DIR = "/content/firms_data" if IN_COLAB else str(pathlib.Path.cwd().parent / "data" / "raw")
DATA_DIR = pathlib.Path(os.environ.get("FIRMS_LOCAL_DIR", DEFAULT_DIR))
BASE = "https://firms.modaps.eosdis.nasa.gov/data/country"

def get_file(inst, year):
    name = f"{inst}_{year}_{COUNTRY}.csv"
    path = DATA_DIR / inst / name
    if path.exists():
        return path
    if "FIRMS_LOCAL_DIR" in os.environ:
        return None  # offline mode: only use files already on disk
    r = requests.get(f"{BASE}/{inst}/{year}/{name}", timeout=120)
    if r.status_code != 200:
        return None  # no file for this sensor and year
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(r.content)
    return path

files = {}
for inst in INSTRUMENTS:
    got = [p for y in YEARS if (p := get_file(inst, y)) is not None]
    files[inst] = got
    print(f"{inst:12} {len(got):2} yearly files")

## 3. One table for both sensors

MODIS and VIIRS name some columns differently and give confidence in different ways (MODIS 0 to 100, VIIRS low / nominal / high). Here both get the same column names, and MODIS confidence is turned into the same three classes using the thresholds in the MODIS user guide (Table 11): below 30 = low, 30 to 79 = nominal, 80 and up = high.

In [ ]:
import pandas as pd

def load(inst):
    if not files[inst]:
        return pd.DataFrame()
    df = pd.concat((pd.read_csv(p) for p in files[inst]), ignore_index=True)
    df = df.rename(columns={"brightness": "bright_main", "bright_t31": "bright_second",
                            "bright_ti4": "bright_main", "bright_ti5": "bright_second"})
    df["sensor"] = INSTRUMENTS[inst]
    df["family"] = "MODIS" if inst == "modis" else "VIIRS"
    if inst == "modis":
        df["confidence_pct"] = pd.to_numeric(df["confidence"], errors="coerce")
        df["confidence_class"] = pd.cut(df["confidence_pct"], [-1, 29.999, 79.999, 100],
                                        labels=["low", "nominal", "high"]).astype(str)
    else:
        df["confidence_pct"] = pd.NA
        df["confidence_class"] = df["confidence"].map({"l": "low", "n": "nominal", "h": "high"})
    return df

fires = pd.concat([load(i) for i in INSTRUMENTS], ignore_index=True)
fires["acq_date"] = pd.to_datetime(fires["acq_date"])
hhmm = fires["acq_time"].astype(int).astype(str).str.zfill(4)
fires["acq_datetime_utc"] = fires["acq_date"] + pd.to_timedelta(hhmm.str[:2].astype(int), unit="h") \
                                              + pd.to_timedelta(hhmm.str[2:].astype(int), unit="m")
fires["year"] = fires["acq_date"].dt.year
fires["month"] = fires["acq_date"].dt.month

COLUMNS = ["sensor", "family", "satellite", "latitude", "longitude", "acq_date", "acq_datetime_utc",
           "year", "month", "daynight", "confidence_class", "confidence_pct", "frp",
           "bright_main", "bright_second", "scan", "track", "type", "version"]
fires = fires[COLUMNS]

print("All hotspots:", len(fires))
print(fires.groupby(["sensor", "type"]).size().unstack(fill_value=0))

if VEGETATION_ONLY:
    fires = fires[fires["type"] == 0].copy()
    print("\nAfter keeping type 0 (vegetation) only:", len(fires))
fires.head()

## 4. Monthly counts per sensor (the gap we need to harmonize)

In [ ]:
import plotly.express as px

monthly = (fires.groupby(["sensor", pd.Grouper(key="acq_date", freq="MS")])
                .size().rename("hotspots").reset_index())
fig = px.line(monthly, x="acq_date", y="hotspots", color="sensor",
              title=f"Monthly hotspot counts, {COUNTRY.replace('_', ' ')}",
              labels={"acq_date": "Month", "hotspots": "Hotspots per month"})
fig.show()

overlap = monthly[monthly["acq_date"].dt.year >= 2012].pivot(index="acq_date", columns="sensor", values="hotspots")
print("Average VIIRS-to-MODIS ratio per month over the overlap years (raw, not harmonized):")
print((overlap.drop(columns="MODIS") .div(overlap["MODIS"], axis=0)).replace([float("inf")], pd.NA).mean().round(2))

## 5. Fire calendar (MODIS only, raw)

A first look at the calendar the app will show: years down, months across, colour = hotspots. This is MODIS alone, before harmonization.

In [ ]:
cal = (fires[fires["sensor"] == "MODIS"].groupby(["year", "month"]).size()
            .unstack(fill_value=0).reindex(columns=range(1, 13), fill_value=0))
cal.columns = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
fig = px.imshow(cal, color_continuous_scale="cividis", aspect="auto",
                labels={"x": "Month", "y": "Year", "color": "Hotspots"},
                title="MODIS hotspots per month (raw)")
fig.show()

## 6. Save the merged table

In [ ]:
OUT_DIR = DATA_DIR if IN_COLAB else DATA_DIR.parent / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
out = OUT_DIR / f"firms_merged_{COUNTRY}.csv"
fires.to_csv(out, index=False)
print("Saved", out, f"({len(fires):,} rows)")

try:
    from google.colab import files as colab_files  # only works inside Colab
    # colab_files.download(str(out))  # remove the # to download the CSV to your laptop
except ImportError:
    pass

## Next

- **Phase 4, harmonization:** compare MODIS and VIIRS over 2012 to 2025 (same area, same months) and build the conversion so the sensor of origin no longer matters.
- Write what you tried and found in the Claude Doc (Session log + Messages), not only here.